# $Bx$ component as Ito's process 
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as ple
import plotly.subplots as sp
from magfield.visual.approximation import harmonic_approximation
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
import pickle
from tqdm.notebook import tqdm


# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

In [ ]:
import os

out_dir = "/home/oplora/Desktop/Diplomanew"
# Create dir if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

## Calculate $a(t), \space b(t)$

In [ ]:
# Load up gaussian mixture model for component Bx
series_name = "Bx"
# with open(f"data/d{series_name}_4300_4.pkl", "rb") as f:
with open(f"data/d{series_name}_4320_3.pkl", "rb") as f:
    gmm = pickle.load(f)

Parameters of Ito's equation are calculated in the next way:

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
b(t) = \sum_{k=1}^{K}{p_k b_k},
$$
where $K$ is a number of mixture components

In [ ]:
from magfield.em.auxiliary import smooth

p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]
K = gmm["num_comp"]
nc = gmm["num_comp"]

# smoothcoef = False
smoothcoef = True

coef_a = np.sum(p * a, axis=0)
coef_b = np.sum(p * b, axis=0)

if smoothcoef:
    # smooth_N = int(0.05 * len(coef_a))
    smooth_N = 60 * 12  # * 7
    coef_a = smooth(coef_a, smooth_N)
    coef_b = smooth(coef_b, smooth_N)

limit = sum(gmm["dates"] < pd.to_datetime("2020-02-01 00:00:00"))

In [ ]:
# Create subplots
# fig = sp.make_subplots(rows=2, cols=1,
#                     subplot_titles=(rf"Ito's coefficient $a(t) = \sum_{{k=1}}^{K}{{p_k a_k}}$",
#                                     rf"Ito's coefficient $b(t) = \sum_{{k=1}}^{K}{{p_k b_k}}$"))
fig = sp.make_subplots(rows=2, cols=1, x_title="Дата", y_title="Значение")

x = gmm["dates"][:limit]
# Add traces for the first plot (coef_a)
fig.add_trace(
    go.Scatter(x=x, y=coef_a[:limit], mode="lines", name="coef_a"), row=1, col=1
)
# Add traces for the second plot (coef_b)
fig.add_trace(
    go.Scatter(x=x, y=coef_b[:limit], mode="lines", name="coef_b"), row=2, col=1
)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"
# fig.update_layout(height=600,width=1800, title_text=f"Ito's Coefficients for d{series_name}" + smooth_title, showlegend=False)
fig.update_layout(height=500, width=1200, showlegend=False)
# Show the plot
fig.show()
fig.write_image(f"{out_dir}/d{series_name}_Ito_coeffs{smooth_file}.png")

# Correlation plots

In [ ]:
correlation_window = (gmm["window"]["size"], gmm["window"]["step"])

#### Correlation $a(t)$, $b(t)=\sum_{j=1}^Kp_jb_j$

In [ ]:
correlation = []
for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
fig = ple.line(x=gmm["dates"][:limit], y=correlation[:limit])

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    xaxis=dict(title=dict(text="Дата")),
    yaxis=dict(title=dict(text="Значение")),
)
fig.write_image(
    f"{out_dir}/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
# title = rf"""$\text{{Correlation between }} a(t), b(t)=
#     \sum_{{j=1}}^{gmm["num_comp"]}p_jb_j \text{{ on window size }}
#     {correlation_window[0]} \text{{ minutes}}$
#     """
# cp = long_plot(correlation, title, dates=gmm["dates"], n=12)
# cp = long_plot(correlation,'', dates=gmm["dates"], n=6, showlegend=False)
# cp.show()
# cp.write_image(f"{out_dir}/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}.png")

In [ ]:
# # Create a histogram plot for a correlation of coef_a and coef_b using plolty express

# percent = np.count_nonzero(np.array([abs(x) for x in correlation]) > 0.5) / len(correlation)
# fig = ple.histogram(
#     x=correlation,
#     nbins=10,
#     # labels={"x": "Correlation", "y": "percents"},
#     # title=f"Percent of values higher than 0.5 for correlation a(t), b(t) of d{series_name} - {round(percent,3)}",
#     title=f"Процент значений больших 0.5 по абсолютному значению - {round(percent,3)}",
#     histnorm="percent",
#     labels={"x": "Значения корреляции", "y": "Процент значений"},
#     height=900,
#     width=1200)
# # fig.show()
# # fig.write_image(f"{out_dir}/d{series_name}_histogram_corr{nc}_{correlation_window[0]}{smooth_file}.png")

# Analysis of $a(t)$ and $b(t)$ relation

## Trigonometric approximation

### Static harmonics for all data

In [ ]:
hn = 24

In [ ]:
fig, resids, paramA = harmonic_approximation(
    data=coef_a,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    # title=f"d{series_name} - a(t)",
    # limit=limit*3
)
fig.show()
fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_approx_a_{hn}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
fig, resids, paramB = harmonic_approximation(
    # data=correlation,
    data=coef_b,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    # title=f"d{series_name} - b(t)",
)
# # fig.show()
# # fig.write_image(f"{out_dir}/d{series_name}_harmonic_approx_b_{hn}_{correlation_window[0]}{smooth_file}.png")

In [ ]:
# fig, resids, paramC = harmonic_approximation(
#     data=correlation,
#     time=gmm["dates"][: len(correlation)],
#     harmonics_num=hn,
#     # title=f"d{series_name} - correlation a(t), b(t)",
# )
# # fig.show()
# # fig.write_image(f"{out_dir}/d{series_name}_harmonic_approx_corr_{hn}_{correlation_window[0]}{smooth_file}.png")

### Parameters of harmonics

In [ ]:
from sys import stdout

harmonicsA = pd.DataFrame(paramA, columns=["Amplitude", "Frequency", "Phase"])
harmonicsA["Frequency"] = harmonicsA["Frequency"] * (60 * 24)  # convert to days
harmonicsA["Period"] = 1 / harmonicsA["Frequency"]
harmonicsA["Amplitude"] = harmonicsA["Amplitude"] * 1000  # convert to pico-Tesla
harmonicsA = harmonicsA.reindex(columns=["Amplitude", "Frequency", "Period", "Phase"])
# display(harmonicsA)
harmonicsA.to_latex(stdout, float_format="%.3f")

# Forecasting

In [ ]:
train_size = int(len(correlation) * 0.95)

In [ ]:
fig, resids, paramA = harmonic_approximation(
    data=coef_a,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - a(t)",
)

In [ ]:
fig, resids, paramB = harmonic_approximation(
    # data=correlation,
    data=coef_b,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - b(t)",
)

In [ ]:
fig, resids, paramC = harmonic_approximation(
    data=correlation,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - correlation a(t), b(t)",
)

In [ ]:
test_data_a = coef_a[train_size + 1 : len(correlation)]
test_data_b = coef_b[train_size + 1 : len(correlation)]
test_data_corr = correlation[train_size + 1 : len(correlation)]

sinA, sinB, sinC = [], [], []
for t in tqdm(range(train_size + 1, len(correlation))):
    sa, sb, sc = 0, 0, 0
    for harm in range(hn):
        sa += paramA[harm][0] * np.sin(paramA[harm][1] * t + paramA[harm][2])
        sb += paramB[harm][0] * np.sin(paramB[harm][1] * t + paramB[harm][2])
        sc += paramC[harm][0] * np.sin(paramC[harm][1] * t + paramC[harm][2])
    sinA.append(sa)
    sinB.append(sb)
    sinC.append(sc)

In [ ]:
df = pd.DataFrame(columns=["mse", "mae"], index=["a_t", "b_t", "corr"])

df.loc["a_t", "mse"] = np.mean((np.array(sinA) - np.array(test_data_a)) ** 2)
df.loc["a_t", "mae"] = np.mean(np.abs(np.array(sinA) - np.array(test_data_a)))
df.loc["b_t", "mse"] = np.mean((np.array(sinB) - np.array(test_data_b)) ** 2)
df.loc["b_t", "mae"] = np.mean(np.abs(np.array(sinB) - np.array(test_data_b)))
df.loc["corr", "mse"] = np.mean((np.array(sinC) - np.array(test_data_corr)) ** 2)
df.loc["corr", "mae"] = np.mean(np.abs(np.array(sinC) - np.array(test_data_corr)))

df

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(go.Scatter(x=x, y=sinA, mode="lines", name="coef_a"))
# Add traces for the second plot (coef_b)
fig.add_trace(go.Scatter(x=x, y=test_data_a, mode="lines", name="coef_b"))

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["a_t"]
mse = df["mse"]["a_t"]

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    xaxis=dict(title=dict(text="Дата")),
    yaxis=dict(title=dict(text="Значение")),
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)
fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_a_{hn}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(go.Scatter(x=x, y=sinB, mode="lines", name="coef_a"))
# Add traces for the second plot (coef_b)
fig.add_trace(go.Scatter(x=x, y=test_data_b, mode="lines", name="coef_b"))

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["b_t"]
mse = df["mse"]["b_t"]

fig.update_layout(
    height=900,
    width=1200,
    showlegend=False,
    xaxis=dict(title=dict(text="Дата")),
    yaxis=dict(title=dict(text="Значение")),
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)

fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_b_{hn}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(go.Scatter(x=x, y=sinC, mode="lines", name="coef_a"))
# Add traces for the second plot (coef_b)
fig.add_trace(go.Scatter(x=x, y=test_data_corr, mode="lines", name="coef_b"))

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["corr"]
mse = df["mse"]["corr"]

fig.update_layout(
    height=900,
    width=1200,
    showlegend=False,
    xaxis=dict(title=dict(text="Дата")),
    yaxis=dict(title=dict(text="Значение")),
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)


fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_corr_{hn}_{correlation_window[0]}{smooth_file}.png"
)